In [7]:
# Adnan Muminhodzic COMP 271 S26 week11.py 4/8/2026
from __future__ import annotations

class ProbingHashTable:
    """Hash table demonstrating linear vs. quadratic probing."""

    # Consstants for default values and error messages
    _DEFAULT_CAPACITY: int = 11
    _PROBE_MODES: tuple[str, str] = ("linear", "quadratic")
    _DEFAULT_PROBING_MODE: str = _PROBE_MODES[0]
    _ERROR_MODE: str = f"mode must be one of {', '.join(_PROBE_MODES)}"
    _ERROR_FULL: str = "Hash table is full"

    def __init__(
        self, capacity: int = _DEFAULT_CAPACITY, mode: str = _DEFAULT_PROBING_MODE
    ):
        """Initialize the hash table with the given capacity and probing mode."""
        if mode not in self._PROBE_MODES:
            # If the mode is not valid,
            # use default mode.
            mode = self._DEFAULT_PROBING_MODE
        self._capacity = capacity
        self._mode = mode
        # Each slot is a string or None
        self._table: list[str | None] = [None] * capacity
        self._size = 0

    def _hash(self, key: int) -> int:
        """Convert an integer value to an index in the underlying table."""
        return key % self._capacity

    def _probe(self, position: int, attempt: int) -> int:
        """"""
        if self._mode == self._DEFAULT_PROBING_MODE:  # linear
            return (position + attempt) % self._capacity
        else:  # quadratic
            return (position + attempt * attempt) % self._capacity

    def load_factor(self) -> float:
        """Return the current load factor of the hash table."""
        return self._size / self._capacity

    def insert(self, value: str) -> list[int]:
        """Insert key-value pair. Returns list of indices probed."""
        # Initialize list to track probe indices
        probes: list[int] = []
        # Check if the hash table has space for a new entry.
        # If not, we will not attempt to insert and will return
        # an empty list of probes.
        if self._size < self._capacity:
            # Generate a non-negative integer key from the value
            key: int = abs(hash(value))
            # Convert the key to an index in the underlying table
            index: int = self._hash(key)
            # Attempt to insert the key-value pair, probing for an empty
            # slot.
            i: int = 0
            insertion_successful: bool = False
            # Loop to attempt to find an empty slot. The loop ends as soon as
            # we have probed the entire table or successfully inserted the
            # value.
            while i < self._capacity and not insertion_successful:
                # Location in the underlying table to check
                probe_index: int = self._probe(index, i)
                # Record the location we are probing
                probes.append(probe_index)
                # Obtain the contents at the location we are probing
                slot: str | None = self._table[probe_index]
                # Determine if we can insert the value at the location we are probing.
                # If insertion is successful, we will exit the loop. If not, we will
                # try the next probe index in the next iteration of the loop.
                insertion_successful = slot is None or slot == value
                if insertion_successful:
                    # The slot is empty or contains the same value already,
                    # so we can insert the value pair here (or update the existing
                    # value if it is the same).
                    self._table[probe_index] = value
                    if slot is None:
                        # Update the size of the hash table if we added
                        # this value for the first time, ie, we have not
                        # overwriten it.
                        self._size += 1
                # If the slot is not empty, we will try the next probe index
                # in the next iteration of the loop.
                i += 1
        return probes

    def contains(self, value: str) -> bool:
        """Check if value is in the hash table."""
        # Generate a non-negative integer key from the value
        key: int = abs(hash(value))
        # Convert the key to an index in the underlying table
        index: int = self._hash(key)
        # Loop to probe for the value. The loop ends as soon as we have probed the
        # entire table or found the value.
        found: bool = False
        i: int = 0
        # Loop ends when we have probed the entire table or found the value
        while i < self._capacity and not found:
            probe_index: int = self._probe(index, i)
            slot: str | None = self._table[probe_index]
            if slot is not None:
                found = slot == value
            i += 1
        return found

    # Formating constants for display
    _FMT_HEADER: str = f"{'Idx':<5}{'Content'}"
    _FMT_EMPTY: str = "---"
    _FMT_SLOT: str = "{idx:5} -> {content}"
    _FMT_HORIZONTAL: str = "-" * 20

    def display(self) -> str:
        """Return a string representation of the hash table."""
        lines: list[str] = [self._FMT_HEADER]
        lines.append(self._FMT_HORIZONTAL)
        for i in range(self._capacity):
            slot: str | None = self._table[i]
            content: str = (
                self._FMT_SLOT.format(idx=i, content=slot) if slot else self._FMT_EMPTY
            )
            lines.append(content)
        return "\n".join(lines)
    
    def measure_probing(self) -> list[list[float]]:
        """Measures average probes per insertion as a function of alpha."""
        hash_results: list = [] # This is the list for (load_factor, average_probes).
        TOTAL_PROBES: int = 0
        _MULTIPLE_VALUE: int = 100000 # This value is for generating six digit strings to be random.
        _NEGATIVE_ONE: int = -1

        # Defining our "found" variable we call "works".
        works = True
        # Add strings until the insertion fails in finding an available slot within bounds.
        while works and self._size < self._capacity:
            probe_list = self.insert(str(_MULTIPLE_VALUE))
            NUMBER_OF_PROBES: int = len(probe_list)

            # Once we hit the capacity limit, stop performing insertions.
            if self._table[probe_list[_NEGATIVE_ONE]] is None:
                works = False

            if works:
                TOTAL_PROBES += NUMBER_OF_PROBES
                hash_results += [[self.load_factor(), TOTAL_PROBES / self._size]]
                _MULTIPLE_VALUE += 1
            
        return hash_results

# Test code for linear and quadratic data respectively.
_DEFAULT_CAPACITY: int = 11
_PROBE_MODES: tuple[str, str] = ("linear", "quadratic")
_DEFAULT_PROBING_MODE: str = _PROBE_MODES[0]
new_line: str = "\n"
    
linear_table = ProbingHashTable(_DEFAULT_CAPACITY, _DEFAULT_PROBING_MODE)
linear_data = linear_table.measure_probing()

quadratic_table = ProbingHashTable(_DEFAULT_CAPACITY, _PROBE_MODES[1])
quadratic_data = quadratic_table.measure_probing()

print(quadratic_data,new_line)
print(linear_data)


[[0.09090909090909091, 1.0], [0.18181818181818182, 1.0], [0.2727272727272727, 1.0], [0.36363636363636365, 1.0], [0.45454545454545453, 1.0], [0.5454545454545454, 1.0], [0.6363636363636364, 1.0], [0.7272727272727273, 1.25], [0.8181818181818182, 1.7777777777777777], [0.8181818181818182, 3.0], [0.8181818181818182, 4.222222222222222], [0.9090909090909091, 4.0], [1.0, 4.090909090909091]] 

[[0.09090909090909091, 1.0], [0.18181818181818182, 1.0], [0.2727272727272727, 1.0], [0.36363636363636365, 1.0], [0.45454545454545453, 1.0], [0.5454545454545454, 1.0], [0.6363636363636364, 1.0], [0.7272727272727273, 1.25], [0.8181818181818182, 1.5555555555555556], [0.9090909090909091, 1.7], [1.0, 2.5454545454545454]]


In [6]:
random_string_assigner(_LENGTH)

'mdneof'